In [1]:
import pandas as pd
import numpy as np
import glob
import os

from sklearn.preprocessing import MinMaxScaler

In [2]:
all_files = glob.glob(
    "../data/raw/*.csv"
)

len(all_files)

37

In [3]:
def load_gdp_file(path):

    df = pd.read_csv(path)

    # clean column names
    df.columns = df.columns.str.strip()


    # check required column
    if "Description" not in df.columns:
        print("Skipped (no Description):", path)
        return pd.DataFrame()


    df["Description"] = (
        df["Description"]
        .astype(str)
        .str.strip()
    )


    gdp = df[
        df["Description"] == "GDP (in Rs. Cr.)"
    ].copy()


    growth = df[
        df["Description"] == "Growth Rate % (YoY)"
    ].copy()


    if gdp.empty or growth.empty:
        print("Skipped (missing GDP/Growth):", path)
        return pd.DataFrame()


    gdp.drop(
        columns=["Description"],
        inplace=True
    )

    growth.drop(
        columns=["Description"],
        inplace=True
    )


    def fix_year(y):

        y = str(y)
        start = int(y.split("-")[0])

        if start >= 1900:
            return start

        if start >= 90:
            return 1900 + start

        return 2000 + start


    gdp["Year"] = gdp["Year"].apply(fix_year)
    growth["Year"] = growth["Year"].apply(fix_year)



    gdp_long = gdp.melt(
        id_vars=["Year"],
        var_name="District",
        value_name="GDP"
    )


    growth_long = growth.melt(
        id_vars=["Year"],
        var_name="District",
        value_name="GrowthRate"
    )


    final = pd.merge(
        gdp_long,
        growth_long,
        on=["Year","District"],
        how="outer"
    )


    return final

In [4]:
def get_state_name(filename):

    name = os.path.basename(filename)

    name = name.replace("gdp_", "")
    
    # remove numbering
    name = name.replace("1","")
    name = name.replace("2","")
    name = name.replace(".csv","")

    return name.strip()

In [5]:
all_states=[]

for file in all_files:

    temp=load_gdp_file(file)

    temp["State"]=get_state_name(file)

    all_states.append(temp)


india_df=pd.concat(
    all_states,
    ignore_index=True
)

Skipped (no Description): ../data/raw\elementary_2015_16.csv
Skipped (missing GDP/Growth): ../data/raw\gdp_Assam2.csv
Skipped (no Description): ../data/raw\india-districts-census-2011.csv
Skipped (no Description): ../data/raw\india_census_housing-hlpca-full.csv


In [6]:
india_df.shape
india_df.head()

,State,Year,District,GDP,GrowthRate
0,AndhraPradesh,1999.0,Adilabad,3463.28,NaN
1,AndhraPradesh,1999.0,Anantapur,4728.59,NaN
2,AndhraPradesh,1999.0,Chittoor,5395.57,NaN
3,AndhraPradesh,1999.0,Godavari East,9507.67,NaN
4,AndhraPradesh,1999.0,Godavari West,7398.0,NaN


In [7]:
india_df.isnull().sum()

State           0
Year            0
District        0
GDP            60
GrowthRate    910
dtype: int64

In [8]:
india_df["GrowthRate"].fillna(
    0,
    inplace=True
)

C:\Users\Maria\AppData\Local\Temp\ipykernel_54260\1457525711.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  india_df["GrowthRate"].fillna(


0              0
1              0
2              0
3              0
4              0
          ...   
6241    5.595392
6242    6.115359
6243    7.115823
6244   -0.274739
6245    8.438402
Name: GrowthRate, Length: 6246, dtype: object

In [9]:
india_df["Year"] = (
    india_df["Year"]
    .astype(int)
)


india_df["GDP"] = (
    india_df["GDP"]
    .astype(str)
    .str.replace(",","")
    .astype(float)
)


india_df["GrowthRate"] = (
    india_df["GrowthRate"]
    .astype(float)
)

india_df = india_df[
    india_df["GDP"] > 0
]

In [10]:
scaler = MinMaxScaler()

india_df["GDP_score"] = (
    scaler.fit_transform(
        india_df[["GDP"]]
    )
)

india_df["Growth_score"] = (
    scaler.fit_transform(
        india_df[["GrowthRate"]]
    )
)

india_df["DDI"] = (
    0.6 * india_df["GDP_score"]
    +
    0.4 * india_df["Growth_score"]
)

In [11]:
india_df["Rank"] = (
    india_df
    .groupby("Year")["DDI"]
    .rank(
        ascending=False,
        method="dense"
    )
)

In [12]:
def category(score):

    if score >= 0.75:
        return "High Development"

    elif score >= 0.45:
        return "Medium Development"

    else:
        return "Low Development"


india_df["Category"] = (
    india_df.DDI.apply(category)
)

In [13]:
state_summary = (
    india_df
    .groupby("State")
    .agg(
        Avg_DDI=("DDI","mean"),
        Avg_GDP=("GDP","mean"),
        Avg_Growth=("GrowthRate","mean"),
        Districts=("District","nunique")
    )
    .reset_index()
)

In [14]:
india_df.to_csv(
"../data/processed/district_development_index.csv",
index=False
)
state_summary.to_csv(
"../data/processed/state_summary.csv",
index=False
)

In [15]:
india_df.shape
india_df.head()
india_df.sort_values(
"DDI",
ascending=False
).head()

,State,Year,District,GDP,GrowthRate,GDP_score,Growth_score,DDI,Rank,Category
3186,Maharashtra,2012,Mumbai,191910.0,7.630788,1.000000,0.422616,0.769046,1.0,High Development
3050,Maharashtra,2008,Mumbai,153339.0,63.364691,0.798988,0.650723,0.739682,1.0,Medium Development
3152,Maharashtra,2011,Mumbai,178304.0,6.187691,0.929092,0.416710,0.724139,1.0,Medium Development
3118,Maharashtra,2010,Mumbai,167914.0,9.455830,0.874945,0.430085,0.697001,1.0,Medium Development
3084,Maharashtra,2009,Mumbai,153408.0,0.044998,0.799347,0.391569,0.636236,1.0,Medium Development
